In [1]:
import numpy as np
import pandas as pd

# import matplotlib.pyplot as plt
# import seaborn as sns

from datetime import date, datetime, timedelta

import os

# Задание 1. Сессии

## Условие

Есть данные следующего вида:

In [2]:
sessions = pd.read_csv('./data/sessions.csv')

sessions

,user_id,event_datetime,duration_sec
0,1,2023-10-01 15:12:06,234
1,2,2023-10-01 23:43:17,187
2,2,2023-10-22 11:12:00,87
3,1,2023-11-15 15:12:24,3
4,1,2023-12-12 00:17:06,28
5,1,2023-12-14 00:25:06,465
6,3,2023-10-05 08:45:30,120
7,2,2023-11-01 14:22:10,300
8,3,2023-11-10 09:15:45,150
9,1,2023-11-20 16:30:00,200


Необходимо написать такой SQL запрос, который выведет пользователя с максимальным промежутком между сессиями. Синтаксис - **Clickhouse SQL**.

## Решение

### Старое

In [3]:
# os.system("curl https://clickhouse.com/ | sh")

query = """
with
    src as (
        select
            user_id,
            event_datetime                           as session_datetime
            --addSeconds(event_datetime, duration_sec) as session_datetime
        from file('./data/sessions.csv', CSVWithNames) as sns
        where
            duration_sec > 0 --Assume it a check for a legit session.
        group by
            1, 2
        order by
            1, 2
    ),

    final as (
        select
            user_id,

            groupArray(session_datetime)           as dt_arr,
            arrayPopFront(arrayDifference(dt_arr)) as diff_arr,
            
            --arrayMax(diff_arr)                     as agg_diff, --1
            --arrayAvg(diff_arr)                     as agg_diff, --3
            --arrayReduce('median', diff_arr)        as agg_diff, --3
            --arraySum(diff_arr)                     as agg_diff, --1
            arrayMin(diff_arr)                     as agg_diff, --3

            row_number() over (
                order by agg_diff desc, user_id asc
            )                                      as rn
        from src
        group by
            1
    )

select user_id
from final
where rn = 1
format CSVWithNames
"""

cmd = """
./clickhouse local -q "
{}
"
""".format(query)
 # > result.csv

os.system(cmd)

"user_id"
3


0

Решение через AWS Redshift строилось бы на
```
lead(event_datetime) over (
    partition by user_id
    order by event_datetime
) as next_event_datetime
```

Но учитывая, что необходимо использовать синтаксис __ClickHouse__, решаем через массивы:
1. Фильтруем невалидные значения, дедуплицируем и сортируем timestamps для будущего массива.
    1. `event_datetime` может быть либо концом ивента (тогда для `session_datetime` просто берем `event_datetime`), либо его началом (тогда добавляем к нему `duration_sec`). На тестовых данных ответы одинаковы.
3. Собираем массив, трансформируем его в first differences (без нулевого/первого 0), применяем агрегатную функцию поверх:
    1. Задача требует "пользователя с __максимальным промежутком между сессиями__", так что берём `max()` среди всех промежутков между сессиями одного пользователя.
    2. Возможны и другие осмысленные агрегации (для слегка по-другому сформулированной задачи: "__максимальным__ _средним/медианным/суммарным/или_даже_минимальным_ промежутком между сессиями"), в комментариях указан результат для них на тестовых данных.
4. Ранжируем по значению агрегата по убыванию: можно использовать `rank()` вместо `row_number()` (на случай ничьей), но в задаче требуется вывести "__пользователя__ с максимальным промежутком между сессиями", так что обеспечим единственность с помощью `row_number()`, а стабильность расчета дополнительным `order by user_id`.

### Новое

>в 1 задаче можно исправить недочет

>формулировка вопроса - время между сессиями. duration_sec там не просто так

>event_datetime - начало сессии \
>код для этой версии - не верный \
>нужно время МЕЖДУ двумя сессиями

In [4]:
# os.system("curl https://clickhouse.com/ | sh")

query = """
with
    src as (
        select
            user_id,
            event_datetime                       as cur_session_start,
            
            max(addSeconds(
                cur_session_start,
                duration_sec
            ))                                   as cur_session_end,
            leadInFrame(event_datetime) over (
                partition by user_id
                order by event_datetime asc
                rows between unbounded preceding and unbounded following
            )                                    as next_session_start,
            
            next_session_start - cur_session_end as time_between_sessions
        from file('./data/sessions.csv', CSVWithNames) as sns
        where
            duration_sec > 0 --Assume it a check for a legit session.
        group by
            1, 2
    ),

    final as (
        select
            user_id,

            max(time_between_sessions) as agg_tbs, --1
            --min(time_between_sessions) as agg_tbs, --3
            --avg(1.0 * time_between_sessions) as agg_tbs, --3
            --median(time_between_sessions) as agg_tbs, --3
            --sum(time_between_sessions) as agg_tbs, --1
            
            row_number() over (
                order by agg_tbs desc, user_id asc
            )                          as rn
        from src
        group by
            1
    )

select user_id
from final
where rn = 1
format CSVWithNames
"""

cmd = """
./clickhouse local -q "
{}
"
""".format(query)
 # > result.csv

os.system(cmd)

"user_id"
1


0

__Исправление:__ промежуток между сессиями вычисляется как разница между концом текущей и началом следующей.
- `event_datetime` — начало сессии.
- `event_datetime + duration_src` — конец сессии.

В таком случае несмотря на то, что синтаксис __ClickHouse__, используем `lead()` (разве что придётся использовать неловкую конструкцию `leadInFrame() over (... rows between unbounded preceding and unbounded following)`).

Комментарии об осмысленной аггрегации и ранжирование те же, что и в прошлом решении.

# Задание 2. Аномалии

## Условие

Есть данные с платежами пользователей следующего вида:

In [5]:
bills = pd.read_csv(
    './data/bills.csv',
    parse_dates=['event_datetime'],
    dtype={
        'payment_method': 'category',
        'country': 'category',
        'status': 'category'
    }
)

bills.head()

,event_datetime,user_id,payment_method,country,amount,status
0,2022-12-01 00:00:00.000000000,9543706,card_processing,AE,10.90,success
1,2022-12-01 00:00:02.592002592,9835632,binance_pay,SA,71.43,cancel
2,2022-12-01 00:00:05.184005184,7526368,card_processing,SA,18.44,cancel
3,2022-12-01 00:00:07.776007776,4572510,card_processing,SA,40.91,cancel
4,2022-12-01 00:00:10.368010368,7832872,binance_pay,SA,23.00,cancel


Полный сэмпл данных в приложенном CSV файле.

Нужно исследовать данные и найти случаи, когда были проблемы с одним или несколькими платежными методами.

Ответ можно предоставить в виде Jupyter Notebook .ipynb файла

## Решение

### Код

In [6]:
bills.info() #No nulls, good.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 6 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   event_datetime  1000000 non-null  datetime64[ns]
 1   user_id         1000000 non-null  int64         
 2   payment_method  1000000 non-null  category      
 3   country         1000000 non-null  category      
 4   amount          1000000 non-null  float64       
 5   status          1000000 non-null  category      
dtypes: category(3), datetime64[ns](1), float64(1), int64(1)
memory usage: 25.8 MB


In [7]:
bills.describe()

,event_datetime,user_id,amount
count,1000000,1.000000e+06,1000000.000000
mean,2022-12-15 23:59:59.999995904,5.491539e+06,50.935791
min,2022-12-01 00:00:00,1.000021e+06,10.000000
25%,2022-12-08 12:00:00,3.251441e+06,14.390000
50%,2022-12-16 00:00:00,5.484437e+06,34.680000
75%,2022-12-23 12:00:00,7.725785e+06,69.400000
max,2022-12-31 00:00:00,9.999985e+06,815.130000
std,NaN,2.590692e+06,49.159379


In [8]:
bills.groupby('status', observed=False)['amount'].describe() #All similar apart from pending max.

,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
cancel,495339.0,50.910096,49.249934,10.0,14.37,34.64,69.34,815.13
pending,156473.0,51.002809,49.126434,10.0,14.39,34.82,69.65,622.05
success,348188.0,50.942226,49.045215,10.0,14.42,34.68,69.39,811.84


In [9]:
bills.groupby('payment_method', observed=False)['amount'].describe() #All similar apart from max-es.

,count,mean,std,min,25%,50%,75%,max
payment_method,,,,,,,,
binance_pay,184078.0,51.005748,49.378256,10.0,14.3700,34.650,69.5000,700.36
card_processing,547390.0,50.877178,49.069210,10.0,14.4000,34.670,69.2700,815.13
jazzcash,57885.0,50.969278,49.336681,10.0,14.4100,34.630,69.3700,591.93
nagad,32965.0,50.950174,49.148393,10.0,14.3000,34.740,69.3800,456.98
paytm,32736.0,50.957367,49.051354,10.0,14.4000,34.805,69.6900,622.05
pix,26950.0,51.337406,49.654032,10.0,14.4100,34.830,69.9700,526.34
skrill,99472.0,50.947755,49.057809,10.0,14.4275,34.800,69.5000,583.69
upi,18524.0,51.155713,49.123222,10.0,14.3000,34.610,70.4625,496.68


In [10]:
bills.groupby('country', observed=False)['amount'].describe() #Same.

,count,mean,std,min,25%,50%,75%,max
country,,,,,,,,
AE,200510.0,50.873197,48.969078,10.0,14.3600,34.60,69.4600,583.22
BD,79526.0,50.991196,49.398827,10.0,14.3100,34.59,69.2975,700.36
BR,50947.0,51.249549,49.650889,10.0,14.4700,34.89,69.7450,556.75
IN,99048.0,50.855936,48.879063,10.0,14.3400,34.63,69.4925,811.84
PK,119387.0,50.907120,49.213584,10.0,14.3700,34.64,69.3800,606.68
SA,400128.0,50.961179,49.200333,10.0,14.4100,34.74,69.3500,815.13
TR,30176.0,50.953652,49.407529,10.0,14.5300,34.62,69.5300,542.41
ZA,20278.0,50.580435,48.724144,10.0,14.3825,34.71,68.8700,539.65


In [11]:
bills_stats = (
    bills
    .groupby(['country', 'payment_method', 'status'], observed=True)
    ['amount'].describe()
    .reset_index()
    .sort_values(['country', 'payment_method', 'status'])
)

pd.options.display.max_rows = bills_stats.shape[0]
bills_stats

,country,payment_method,status,count,mean,std,min,25%,50%,75%,max
0,AE,binance_pay,cancel,46882.0,51.000850,49.114783,10.00,14.4300,34.740,69.6300,515.48
1,AE,card_processing,cancel,60228.0,50.426908,48.540011,10.00,14.3600,34.250,68.7525,575.57
2,AE,card_processing,pending,595.0,51.665328,48.709630,10.00,14.1550,34.630,71.3900,300.08
3,AE,card_processing,success,69234.0,51.105877,49.233498,10.00,14.2300,34.710,69.7900,550.82
4,AE,skrill,pending,21507.0,51.056879,49.107822,10.00,14.5950,34.880,70.0300,583.22
5,AE,skrill,success,2064.0,51.049283,47.810847,10.00,14.8100,35.550,68.9725,318.25
6,BD,binance_pay,cancel,9781.0,50.998183,49.757583,10.00,14.3500,34.210,68.5900,700.36
7,BD,binance_pay,pending,16.0,43.140000,40.152011,10.00,10.0000,37.550,56.4875,141.94
8,BD,binance_pay,success,1089.0,53.123747,50.721211,10.00,12.1900,36.320,74.9600,414.09
9,BD,card_processing,cancel,14005.0,51.188492,50.168675,10.00,14.0600,34.670,69.7000,523.29


In [12]:
bills.loc[:, ['amount']].describe().transpose()

,count,mean,std,min,25%,50%,75%,max
amount,1000000.0,50.935791,49.159379,10.0,14.39,34.68,69.4,815.13


Статистически всё преимущественно ровно, некоторое сильные девиации можно объяснить низким числом ивентом (например, `ZA-binance_pay-pending` 75% __94__ против обычного 69, но их всего лишь 5), а некоторые нет (`ZA-binance_pay-pending` 75% __34.7__ из 4389 ивентов)

In [13]:
categories = ('payment_method', 'status', 'country')

for cat in categories:
    print(bills[cat].value_counts(), '\n')

payment_method
card_processing    547390
binance_pay        184078
skrill              99472
jazzcash            57885
nagad               32965
paytm               32736
pix                 26950
upi                 18524
Name: count, dtype: int64 

status
cancel     495339
success    348188
pending    156473
Name: count, dtype: int64 

country
SA    400128
AE    200510
PK    119387
IN     99048
BD     79526
BR     50947
TR     30176
ZA     20278
Name: count, dtype: int64 



In [14]:
bills.loc[:, ['country', 'payment_method']].value_counts().sort_index()

country  payment_method 
AE       binance_pay         46882
         card_processing    130057
         skrill              23571
BD       binance_pay         10886
         card_processing     30232
         nagad               32965
         skrill               5443
BR       binance_pay          5585
         card_processing     15609
         pix                 26950
         skrill               2803
IN       card_processing     40432
         paytm               32736
         skrill               7356
         upi                 18524
PK       binance_pay         14615
         card_processing     39726
         jazzcash            57885
         skrill               7161
SA       binance_pay         94051
         card_processing    258881
         skrill              47196
TR       binance_pay          7176
         card_processing     19448
         skrill               3552
ZA       binance_pay          4883
         card_processing     13005
         skrill               

In [15]:
bills['event_date'] = bills['event_datetime'].dt.date
bills['event_date'].value_counts().sort_index()

event_date
2022-12-01    33334
2022-12-02    33333
2022-12-03    33333
2022-12-04    33334
2022-12-05    33333
2022-12-06    33333
2022-12-07    33334
2022-12-08    33333
2022-12-09    33333
2022-12-10    33333
2022-12-11    33334
2022-12-12    33333
2022-12-13    33333
2022-12-14    33334
2022-12-15    33333
2022-12-16    33333
2022-12-17    33334
2022-12-18    33333
2022-12-19    33333
2022-12-20    33333
2022-12-21    33334
2022-12-22    33333
2022-12-23    33333
2022-12-24    33334
2022-12-25    33333
2022-12-26    33333
2022-12-27    33334
2022-12-28    33333
2022-12-29    33333
2022-12-30    33333
2022-12-31        1
Name: count, dtype: int64

In [16]:
bills.loc[bills['event_date']==date(2022, 12, 31), :]

,event_datetime,user_id,payment_method,country,amount,status,event_date
999999,2022-12-31,4105705,card_processing,SA,129.44,success,2022-12-31


In [17]:
bills_30 = bills.loc[bills['event_date']==date(2022, 12, 30), :].sort_values('event_datetime')

bills_30.head()

,event_datetime,user_id,payment_method,country,amount,status,event_date
966666,2022-12-30 00:00:00.777600777,7915968,binance_pay,PK,10.00,cancel,2022-12-30
966667,2022-12-30 00:00:03.369603369,3510853,card_processing,SA,23.59,cancel,2022-12-30
966668,2022-12-30 00:00:05.961605961,9435216,card_processing,SA,10.00,success,2022-12-30
966669,2022-12-30 00:00:08.553608553,4494745,card_processing,AE,28.64,cancel,2022-12-30
966670,2022-12-30 00:00:11.145611145,9908360,card_processing,AE,14.76,success,2022-12-30


In [18]:
bills_30.tail()

,event_datetime,user_id,payment_method,country,amount,status,event_date
999994,2022-12-30 23:59:47.039987040,4434027,card_processing,SA,44.87,cancel,2022-12-30
999995,2022-12-30 23:59:49.631989632,9064597,card_processing,SA,43.76,success,2022-12-30
999996,2022-12-30 23:59:52.223992224,3323918,pix,BR,159.32,success,2022-12-30
999997,2022-12-30 23:59:54.815994816,9937162,skrill,SA,127.25,pending,2022-12-30
999998,2022-12-30 23:59:57.407997408,9489398,card_processing,SA,10.00,cancel,2022-12-30


Видимо, всё упало 2022-12-31 (скорее всего, единственная запись в 0:00:00 больше относилась к предыдущему дню).

Посмотрим, единственная ли это проблема. "Нужно видеть не только то, что в данных есть, но и чего в них нет" — проверим, все ли осмысленные комбинации присутствуют.

Я бы хотел __каждый день__ видеть:
- все статусы,
- все пары страна-метод_оплаты (а не просто "все страны и все методы оплаты", потому что некоторые методы могут быть недоступны в некоторых странах).

In [19]:
all_statuses = pd.DataFrame(bills['status'].unique(), columns=['status', ])
all_dates = pd.DataFrame(bills['event_date'].unique(), columns=['event_date', ])

bills_cmb = (
    bills
    .loc[:, ['country', 'payment_method']]
    .value_counts().reset_index()
    .merge(
        right=all_statuses,
        how='cross'
    )
    .merge(
        right=all_dates,
        how='cross'
    )
    .loc[: , ['country', 'payment_method', 'status', 'event_date']]
)

bills_cmb

,country,payment_method,status,event_date
0,SA,card_processing,success,2022-12-01
1,SA,card_processing,success,2022-12-02
2,SA,card_processing,success,2022-12-03
3,SA,card_processing,success,2022-12-04
4,SA,card_processing,success,2022-12-05
...,...,...,...,...
2599,ZA,skrill,pending,2022-12-27
2600,ZA,skrill,pending,2022-12-28
2601,ZA,skrill,pending,2022-12-29
2602,ZA,skrill,pending,2022-12-30


In [20]:
bills_all = (
    bills_cmb
    .merge(
        right=bills,
        how='left',
        on=['country', 'payment_method', 'status', 'event_date']
    )
)

bills_all.head()

,country,payment_method,status,event_date,event_datetime,user_id,amount
0,SA,card_processing,success,2022-12-01,2022-12-01 00:00:12.960012960,9019766.0,52.13
1,SA,card_processing,success,2022-12-01,2022-12-01 00:00:25.920025920,3886551.0,35.76
2,SA,card_processing,success,2022-12-01,2022-12-01 00:00:44.064044064,3713155.0,10.00
3,SA,card_processing,success,2022-12-01,2022-12-01 00:01:12.576072576,5481180.0,64.60
4,SA,card_processing,success,2022-12-01,2022-12-01 00:01:15.168075168,8140859.0,324.62


In [21]:
bills_all['has_missing'] = bills_all['amount'].isna()

# bills_all.groupby(['country', 'payment_method', 'status', 'event_date'])['amount'].isna().sum()
bills_all['has_missing'].value_counts()

has_missing
False    1000000
True         554
Name: count, dtype: int64

In [22]:
#Добавили 554 комбинации, которых не было в оригинальном файле.

bills_missing = (
    bills_all
    .groupby(['payment_method', 'status', 'event_date'], observed=True) #'country', 'status',
    ['has_missing'].min()
    .reset_index()
)

#Если оставить 2022-12-31, то смысла в выводе не будет — там не хватает всех.
(
    bills_missing
    .loc[
        (bills_missing['has_missing'] > 0) & (bills_missing['event_date'] != date(2022, 12, 31)),
        'payment_method'
    ].unique()
)

['nagad', 'skrill', 'upi']
Categories (8, object): ['binance_pay', 'card_processing', 'jazzcash', 'nagad', 'paytm', 'pix', 'skrill', 'upi']

In [23]:
#Дальше проверю в Tableau, там удобнее переключать фильтры в UI.
# bills_all.to_csv('./data/bills_all.csv')

### Ответ

__Главная проблема__: всё упало __2022-12-31__ — в эту дату пришёл один платёж, причём он, видимо, задержался с предыдущего дня.

__Другие важные проблемы:__
1. `binance_pay` из `AE` имеет __100%__ отказов на всём промежутке (нет ни `pending`, ни `success` ивентов),  для остальных стран процент `cancel` стабильно __90%__ (что очень плохо).
2. У `nagad` и `skrill` вообще нет `cancel` ивентов на всём промежутке, __90%+__ `pending`. Возможно, это фича (или вообще очень хорошо, не нашёл на их сайтах), но настораживает.
3. У `upi` нет `cancel` ивентов c __2022-12-03__. Так как они когда-то были, гораздо менее вероятно, что это хорошо, гораздо более вероятно, что что-то сломалось и эти ивенты не доходят.

_Менее важные проблемы:_
1. У `binance_pay` ежедневно стабильно около __93%__ ивентов `cancel` — возможно, это фича.
2. У `binance_pay` и `card_processing` почти нет `pending` ивентов, что хорошо, потому что чем меньше, тем лучше, но при этом они есть, так что данные (скорее всего) идут — хотя есть дни, в которые из некоторых стран их не было, например, для `binance_pay`: __BR__ до 2022-12-19; __PK__ между 2022-12-02 и 2022-12-18; __TR__ между 2022-12-09 и 2022-12-21.

Остальные данные, даже если выглядят не особо красиво, соответствуют некоторому паттерну (девиации подневных сумм по статусам от средней суммы по всем дням незначительны) — до __2022-12-31__.

## Решение2

>во 2 немного доделать. статусы можно рассматривать как success и не success, а не 3 варианта \
>pending и cancel - это одно и то же (non-success)

### Код

In [24]:
bills['is_success'] = bills['status'] == 'success'

bills['is_success'].value_counts()

is_success
False    651812
True     348188
Name: count, dtype: int64

In [25]:
bills_stats = (
    bills
    .groupby(['country', 'payment_method', 'is_success'], observed=True)
    ['amount'].describe()
    .reset_index()
    .sort_values(['country', 'payment_method', 'is_success'])
)

pd.options.display.max_rows = bills_stats.shape[0]
bills_stats

,country,payment_method,is_success,count,mean,std,min,25%,50%,75%,max
0,AE,binance_pay,False,46882.0,51.000850,49.114783,10.0,14.4300,34.740,69.6300,515.48
1,AE,card_processing,False,60823.0,50.439023,48.541424,10.0,14.3550,34.250,68.7950,575.57
2,AE,card_processing,True,69234.0,51.105877,49.233498,10.0,14.2300,34.710,69.7900,550.82
3,AE,skrill,False,21507.0,51.056879,49.107822,10.0,14.5950,34.880,70.0300,583.22
4,AE,skrill,True,2064.0,51.049283,47.810847,10.0,14.8100,35.550,68.9725,318.25
5,BD,binance_pay,False,9797.0,50.985350,49.742764,10.0,14.3400,34.230,68.5800,700.36
6,BD,binance_pay,True,1089.0,53.123747,50.721211,10.0,12.1900,36.320,74.9600,414.09
7,BD,card_processing,False,14124.0,51.189124,50.159703,10.0,14.0575,34.700,69.7100,523.29
8,BD,card_processing,True,16108.0,50.691772,48.857772,10.0,14.5100,34.295,68.7925,460.99
9,BD,nagad,False,31445.0,50.981948,49.189687,10.0,14.4100,34.790,69.2300,456.98


После перехода на бинарные статусы все, кроме максимумов (и 75% `BD-binance_pay-True`), стало статистически эквивалентным.

In [26]:
all_successes = pd.DataFrame([True, False], columns=['is_success', ])
# Надежнее было бы взять весь декабрь из скелета календаря, а не (если в них чего-то нет, то упустили бы), но в этом случае разницы нет.
all_dates = pd.DataFrame(bills['event_date'].unique(), columns=['event_date', ])

bills_cmb = (
    bills
    .loc[:, ['country', 'payment_method']]
    .value_counts().reset_index()
    .merge(
        right=all_successes,
        how='cross'
    )
    .merge(
        right=all_dates,
        how='cross'
    )
    .loc[: , ['country', 'payment_method', 'is_success', 'event_date']]
)
bills_cmb

,country,payment_method,is_success,event_date
0,SA,card_processing,True,2022-12-01
1,SA,card_processing,True,2022-12-02
2,SA,card_processing,True,2022-12-03
3,SA,card_processing,True,2022-12-04
4,SA,card_processing,True,2022-12-05
...,...,...,...,...
1731,ZA,skrill,False,2022-12-27
1732,ZA,skrill,False,2022-12-28
1733,ZA,skrill,False,2022-12-29
1734,ZA,skrill,False,2022-12-30


In [27]:
bills_all = (
    bills_cmb
    .merge(
        right=bills,
        how='left',
        on=['country', 'payment_method', 'is_success', 'event_date']
    )
)

bills_all.head()

,country,payment_method,is_success,event_date,event_datetime,user_id,amount,status
0,SA,card_processing,True,2022-12-01,2022-12-01 00:00:12.960012960,9019766.0,52.13,success
1,SA,card_processing,True,2022-12-01,2022-12-01 00:00:25.920025920,3886551.0,35.76,success
2,SA,card_processing,True,2022-12-01,2022-12-01 00:00:44.064044064,3713155.0,10.00,success
3,SA,card_processing,True,2022-12-01,2022-12-01 00:01:12.576072576,5481180.0,64.60,success
4,SA,card_processing,True,2022-12-01,2022-12-01 00:01:15.168075168,8140859.0,324.62,success


In [28]:
bills_all['has_missing'] = bills_all['amount'].isna()

# bills_all.groupby(['country', 'payment_method', 'status', 'event_date'])['amount'].isna().sum()
bills_all['has_missing'].value_counts()

has_missing
False    1000000
True          85
Name: count, dtype: int64

In [29]:
#Добавили 85 комбинаций, которых не было в оригинальном файле.

bills_missing = (
    bills_all
    .groupby(['payment_method', 'is_success', 'event_date'], observed=True) #'country', 'status',
    ['has_missing'].min()
    .reset_index()
)

#Если оставить 2022-12-31, то смысла в выводе не будет — там не хватает всех.
#А если не оставить, новый вывод пустой.
(
    bills_missing
    .loc[
        (bills_missing['has_missing'] > 0) & (bills_missing['event_date'] != date(2022, 12, 31)),
        'payment_method'
    ].unique()
)

[], Categories (8, object): ['binance_pay', 'card_processing', 'jazzcash', 'nagad', 'paytm', 'pix', 'skrill', 'upi']

In [30]:
#Дальше проверю в Tableau, там удобнее переключать фильтры в UI.
#bills_all.to_csv('./data/bills_all2.csv')

### Ответ

__Главная проблема__ не изменилась: всё упало __2022-12-31__ — в эту дату пришёл один платёж, причём он, видимо, задержался с предыдущего дня.

__Другие важные проблемы:__

_Disclaimer:_ Указанные проценты показывают часть суммы `amount` для `success/non-success` от общей суммы, так как это кажется более адекватной мерой, чем `count`. Проценты не отличаются за исключением `skrill` — вместо __90%__ `non-success` будет __91%__.

1. `binance_pay` ежедневно имеет около __93%__ `non-success` ивентов, причем из `AE` — __100%__ на всём промежутке.
2. `jazzcash` имеет __97%__ `non-success` ивентов на всём промежутке (всё из __PK__).
3. `nagad` имеет __95%__ `non-success` ивентов на всём промежутке (всё из __BD__).
4. `skrill` имеет около __90%__ `non-success` ивентов на всём промежутке (много стран, везде приблизительно такой процент).

Из _позитивного_ хочется отметить:
1. `paytm` приемлем — __30%__ `success` (ниже средних по выборке __35%__).
2. `upi` хорош — __41%__ `success` (немногим выше среднего по выборке).
3. `card_processing` и `pix` прекрасны — __53%__ и __52%__ `success` (намного выше среднего по выборке, лучшие здесь).


"Feedback":

>- найдено 2 из 3 аномалий
>- ожидались временные графики
>- ожидалось применение какого-либо алгоритма детекции аномалий
>- задача на sql решена не с первого раза
>
>в целом, решение не плохое и мы предусматриваем погрешности в решении, и на основные наши специализации аналитиков такое подходит,
>но вакансия на платежного аналитика -  она более высокого уровня"

# Бонус. Парадокс Симпсона

## Официальное Условие

Исследуется польза введения новой учебной программы в университете для первокурсников.
Оценка эффективности проводится по результатам одного экзамена в конце года с оценкой сдал/не сдал.
Экзамен не менялся, менялась только учебная программа.
Считается что результативность экзамена - хороший показатель эффективности учебной программы.
Исследование проводится по 2 годам. 1 - до введения. 2 - когда программа была введена.
Оба года принималось одинаковое число студентов.
Среди студентов второго года нет тех кто учился и в первый год.
Предполагается что в конце года все поступившие студенты участвовали в экзамене.
 
Результаты поступили в следующем виде: 
1. Общая успешност сдачи экзамена - снизилась.
2. Успешность сдачи экзамена у мужчин выросла. У женщин - также выросла. 
 
Вопросы: 
1. Как такое может быть?
2. Какие можно сделать выводы о пользе нововведения?
Объяснения нужно дать для условного деканата, который со понятиями статистики и теории вероятностей не знаком.

## Условие

Есть университет, в котором обучаются люди двух классов: A и B. Успех каждого студента определяется результатом экзамена — зачет или незачет. Экзамен является точной оценкой успеваемости студента. Студентов достаточно, чтобы обеспечить статистически значимые результаты.

В прошлом году __60%__ студентов получили зачет.

В этом году университет обновил программу. Процент успеха в обоих классах вырос, но общий показатель упал до __50%__.

1. Как такое возможно? (Нельзя просто назвать имя феномена, нужно объяснить.)
2. Должен ли университет откатиться к прошлой программе или продолжать с текущей?

## Математическая формулировка

[Парадокс Симпсона](https://en.wikipedia.org/wiki/Simpson%27s_paradox) во всей красе _(только этого мало)_. Распишем конверсию из попытки студента сдать экзамен в зачет: каждый студент пытается сдать экзамен, у него или нее это получается в зависимости от класса и программы:

$$ C=\frac{S}{N}=\frac{\sum{S_i}}{\sum{N_i}}=\frac{\sum{C_i \cdot N_i}}{\sum{N_i}}$$

В случае двух классов A и B получаем:

$$ C=\frac{C_A \cdot N_A + C_B \cdot N_B}{N_A + N_B}$$

$$ wlg \; C_A \leq C_B : $$
$$ C_B = C_A + u, \;\; 0 \leq u \leq 1 - C_A $$

Почему $wlg$? Потому что если $C_A \geq C_B$, то мы просто меняем индексы и оказываемся в той же ситуации.

Для простоты положим, что $N$ не меняется между годами ("примем пропускную способность университета приблизительно постоянной"). Тогда:
$$C=\frac{C_A \cdot N_A + C_B \cdot N_B}{N_A + N_B}=\frac{C_A \cdot N_A + (C_A + u) \cdot (N - N_A)}{N_A + N - N_A}=$$
$$=\frac{C_A \cdot N_A + C_A \cdot N + u \cdot N - C_A \cdot N_A - u \cdot N_A}{N}=\frac{C_A \cdot N + u \cdot (N - N_A)}{N}=$$
$$=C_A+\frac{N - N_A}{N} \cdot u$$

Рассмотрим краевые случаи:
$$N_A=N: \;\; C=C_A+\frac{N - N}{N} \cdot u = C_A$$
$$N_A=0: \;\; C=C_A+\frac{N - 0}{N} \cdot u = C_A + u = C_B$$

Получаем, что общая конверсия $C$ принадлежит интервалу $(C_A, C_B)$. Именно __интервалу__, а не отрезку, так как у нас есть два класса с __ненулевыми__ размерами, а особенный случай $C_A=C_B=C$ иррелевантен в этой задаче, потому что невозможно найти такую пару лет ($C^i$ есть конверсия в год $i$), при которых:
$$
\begin{cases}
C_A^2 > C_A^1 \\
C_B^2 > C_B^1 \\
C^2 < C^1 \\
\end{cases}
$$
Потому что это равносильно
$$
\begin{cases}
C^2 > C_A^1 \\
C^2 > C_B^1 \\
C^2 < C^1 \\
\end{cases}
$$
А, как показано выше, $\forall i \; C_A^i \leq C^i \leq C_B^i$, засим получаем противоречие:
- конверсия второго года больше верхней границы конверсии первого года: $C^2 > C_B^1 \geq C^1 \rightarrow C^2 > C^1$,
- но при этом меньше самой конверсии первого года $C^2 < C^1$.

Поэтому в этой задаче:
$$\forall i \;\; C_A^i < C^i < C_B^i$$

Возможна даже ситуация, при которой средние стали хуже в обоих подгруппах, но общее стало лучше — дисбаланс классов позволяет варьировать общую конверсию довольно сильно (хоть и в пределах групповых). Это напомнило мне [видео 3blue1brown "The medical test paradox, and redesigning Bayes' rule"](https://youtu.be/lG4VkPoG3ko?list=PLiAulSm0XXgvCGe63mrAkda9UQ9478YQv&t=902):
![image](./images/b1.jpg)
![image](./images/b2.jpg)
![image](./images/b3.jpg)

Тест влияет на конверсию по классу, а не на общую. И если дисбаланс классов настолько велик, что может обнулить или даже развернуть эффект от смены контроля (программы), то проблема не в контроле, а в дизайне эксперимента: выборе метрики и/или классовом распределении.

## Численные примеры

In [31]:
N = 1_000

def calc_c(c_a, c_b, n_a, N_=N):
    n_b = N_ - n_a
    return (c_a * n_a + c_b * n_b) / N_

def calc_c_u(c_a, u, n_a, N_=N):
    """Calculate via u(plift)."""
    return calc_c(c_a, c_a+u, n_a, N_=N)


P_I = (0.40, 0.80)
P_II = (41/90, 0.90)
P_III = (0.60, 0.95)

cfg_tuples = (
    (*P_I, 500),
    (*P_II, 900),
    (*P_I, 400),
    (*P_II, 400),
    (*P_II, 950),
    (*P_III, 975),
    (*P_III, 500),
    (*P_III, 25)
)

def print_c_results(c_tuples, func):
    for i in range(len(c_tuples)):
        ct=c_tuples[i]
        print('{}: C_A={:.2%}; C_B={:.2%}; N_A={:,}; N_B={:,} -> C={:.2%}'.format(i+1, *ct, N-ct[2], func(*ct)))
        
        if i+1 in (3, 5):
            print('\n')

print_c_results(cfg_tuples, calc_c)

1: C_A=40.00%; C_B=80.00%; N_A=500; N_B=500 -> C=60.00%
2: C_A=45.56%; C_B=90.00%; N_A=900; N_B=100 -> C=50.00%
3: C_A=40.00%; C_B=80.00%; N_A=400; N_B=600 -> C=64.00%


4: C_A=45.56%; C_B=90.00%; N_A=400; N_B=600 -> C=72.22%
5: C_A=45.56%; C_B=90.00%; N_A=950; N_B=50 -> C=47.78%


6: C_A=60.00%; C_B=95.00%; N_A=975; N_B=25 -> C=60.88%
7: C_A=60.00%; C_B=95.00%; N_A=500; N_B=500 -> C=77.50%
8: C_A=60.00%; C_B=95.00%; N_A=25; N_B=975 -> C=94.12%


$1 \rightarrow 2$ — ситуация из условия.


Варианты:
- $2 \rightarrow 3$ — худший сценарий: университет в панике откатывает программу, получает лучшую общую конверсию, и, не обращая внимание на ухудшение в обоих классах, считает, что сделал все правильно (64% > 50%).
- $2 \rightarrow 4$ — хотя если бы мы остались на программе __II__, всё было бы ещё лучше (72% > 64% > 50%)!
- $2 \rightarrow 5$ — или нет... (48% < 50% < 64%) — до тех пор, пока мы не контролируем (дис)баланс классов, мы не можем гарантировать адекватное сравнение.


Даже выдающаяся программа __III__ может оказаться в ситуации, когда она еле лучше оригинальной __I__ ($2 \rightarrow 6$: 61% > 60%), хотя в балансе она намного лучше неё ($2 \rightarrow 7$: 78% > 60%), а в обратном дисбалансе намного-много лучше ($2 \rightarrow 8$: 94% > 60%).

## Предложения

### Другая метрика

Поменяем целевую метрику: рассчитаем общую конверсию как среднее между поклассовыми конверсиями (некоторое "перевзвешивание": теперь оба класса одинаково важны вне зависимости от их размеров).

In [32]:
def calc_c_avg(c_a, c_b, n_a, N_=N):
    return (c_a + c_b) / 2

print_c_results(cfg_tuples, calc_c_avg)

1: C_A=40.00%; C_B=80.00%; N_A=500; N_B=500 -> C=60.00%
2: C_A=45.56%; C_B=90.00%; N_A=900; N_B=100 -> C=67.78%
3: C_A=40.00%; C_B=80.00%; N_A=400; N_B=600 -> C=60.00%


4: C_A=45.56%; C_B=90.00%; N_A=400; N_B=600 -> C=67.78%
5: C_A=45.56%; C_B=90.00%; N_A=950; N_B=50 -> C=67.78%


6: C_A=60.00%; C_B=95.00%; N_A=975; N_B=25 -> C=77.50%
7: C_A=60.00%; C_B=95.00%; N_A=500; N_B=500 -> C=77.50%
8: C_A=60.00%; C_B=95.00%; N_A=25; N_B=975 -> C=77.50%


### Разделение

Другое возможное решение проблемы дисбаланса — разделение (сплитинг). Оставляем обе программы, разделяя на статистически равные подгруппы (как это сделали, например [Morisano, Hirsh, Peterson, Pihl, And Shore](http://individual.utoronto.ca/jacobhirsh/publications/GoalSettingJAP2010.pdf), страница 4/258) — в нашем случае это очень просто, потому что характеристика одна — класс. Тогда можно будет сравнить и погрупповые, и общее, потому что оно хоть и смещённое, но в одну и ту же сторону в обоих случаях.


In [33]:
program_tuples = (
    (P_I, P_II, 500),
    (P_I, P_II, 25),
    (P_I, P_II, 975),
    (P_II, P_I, 100),
    (P_II, P_I, 500),
    (P_II, P_I, 900),
    (P_I, P_III, 10),
    (P_I, P_III, 500),
    (P_I, P_III, 990)
)

def calc_c_split(prog_1, prog_2, n_a, func, N_=N):
    n_a1 = round(n_a / 2)
    n_a2 = n_a - n_a1

    n_b1, n_b2 = round(N_/2 - n_a1), round(N_/2 - n_a2)
    
    return (func(*prog_1, n_a1, N_=N_/2), n_a1, n_b1, func(*prog_2, n_a2, N_=N_/2), n_a2, n_b2)


def print_c_results_split(c_tuples, split_func, func):
    for i in range(len(c_tuples)):
        ct = c_tuples[i]
        res = split_func(*ct, func)
        print("""{}:
P_I:    C_A={:.2%}, C_B={:.2%};
P_II:   C_A={:.2%}, C_B={:.2%};
Sample: N_A={:,}; N_B={:,}
Totals: C_I={:.2%} (split {:,}/{:,}); C_II={:.2%} (split {:,}/{:,})
Comparison (Pre):  P_I < P_II ? A: [{}]; B: [{}]
Comparison (Post): P_I < P_II ? Total: [{}]
        """.format(
                i+1,
                *ct[0],
                *ct[1],
                ct[2],
                N-ct[2],
                *res,
                ct[0][0] < ct[1][0],
                ct[0][1] < ct[1][1],
                res[0] < res[3]
            )
        )
    print('\n')

print('>Function: calc_c')
print_c_results_split(program_tuples, calc_c_split, calc_c)

print('>Function: calc_c_avg')
print_c_results_split(program_tuples, calc_c_split, calc_c_avg)

>Function: calc_c
1:
P_I:    C_A=40.00%, C_B=80.00%;
P_II:   C_A=45.56%, C_B=90.00%;
Sample: N_A=500; N_B=500
Totals: C_I=60.00% (split 250/250); C_II=67.78% (split 250/250)
Comparison (Pre):  P_I < P_II ? A: [True]; B: [True]
Comparison (Post): P_I < P_II ? Total: [True]
        
2:
P_I:    C_A=40.00%, C_B=80.00%;
P_II:   C_A=45.56%, C_B=90.00%;
Sample: N_A=25; N_B=975
Totals: C_I=79.04% (split 12/488); C_II=88.84% (split 13/487)
Comparison (Pre):  P_I < P_II ? A: [True]; B: [True]
Comparison (Post): P_I < P_II ? Total: [True]
        
3:
P_I:    C_A=40.00%, C_B=80.00%;
P_II:   C_A=45.56%, C_B=90.00%;
Sample: N_A=975; N_B=25
Totals: C_I=40.96% (split 488/12); C_II=46.71% (split 487/13)
Comparison (Pre):  P_I < P_II ? A: [True]; B: [True]
Comparison (Post): P_I < P_II ? Total: [True]
        
4:
P_I:    C_A=45.56%, C_B=90.00%;
P_II:   C_A=40.00%, C_B=80.00%;
Sample: N_A=100; N_B=900
Totals: C_I=85.56% (split 50/450); C_II=76.00% (split 50/450)
Comparison (Pre):  P_I < P_II ? A: [False]

Теперь пред- и пост-сравнения всегда сонаправлены, это как раз то, чего мы хотели.

### Вывод

И выбор другой метрики, и разделение подгрупп обеспечивают сонаправленность эффектов, что разрешает парадокс Сиспсона в конкретно взятой задаче.

У обоих предложенных решений могут (и я почти уверен, что будут) проблемы с парами программ, в которой одна __не полностью__ доминирует другую:
$$ P_{II} \;?\; P_{I}:  C_A^{P_{II}} \geq C_A^{P_I}; \;\; C_B^{P_{II}} \leq C_B^{P_{I}} $$
"but that's beyond the scope of this study" (хотя мне кажется, что неплохим решением этой проблемы будет приоритизация классов).


(Учёт confounding variable ChatGPT предлагает проводить через логистическую регрессию.)